In [3]:
!pip install -q beir sentence-transformers transformers faiss-cpu gradio

In [4]:
import numpy as np
import pandas as pd
import torch
import faiss

from beir import util
from beir.datasets.data_loader import GenericDataLoader
from sentence_transformers import SentenceTransformer

In [5]:
url = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/scifact.zip"

data_path = util.download_and_unzip(url, "data")

corpus, queries, qrels = GenericDataLoader(
    data_folder=data_path
).load(split="train")

print("Documents:", len(corpus))
print("Claims:", len(queries))
print("Relevance judgments:", len(qrels))

data/scifact.zip:   0%|          | 0.00/2.69M [00:00<?, ?iB/s]

  0%|          | 0/5183 [00:00<?, ?it/s]

Documents: 5183
Claims: 809
Relevance judgments: 809


In [6]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

Device: cpu


In [7]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    device=device
)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [8]:
def retrieve_candidates(claim, top_k=20):

    query_embedding = model.encode(
        claim,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(
        query_embedding.reshape(1, -1),
        top_k
    )

    candidates = []

    for score, idx in zip(scores[0], indices[0]):
        doc_id = doc_ids[idx]

        candidates.append({
            "doc_id": doc_id,
            "title": corpus[doc_id]["title"],
            "text": corpus[doc_id]["text"],
            "retrieval_score": float(score)
        })

    return candidates

In [9]:
def rerank_documents(claim, candidates, top_k=5):

    pairs = [
        (claim, candidate["title"] + " " + candidate["text"])
        for candidate in candidates
    ]

    scores = reranker.predict(pairs)

    for candidate, score in zip(candidates, scores):
        candidate["rerank_score"] = float(score)

    ranked = sorted(
        candidates,
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    return ranked[:top_k]

In [11]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=device
)

print("Sentence-BERT loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Sentence-BERT loaded successfully!


In [12]:
doc_ids = list(corpus.keys())

documents = [
    corpus[doc_id]["title"] + " " + corpus[doc_id]["text"]
    for doc_id in doc_ids
]

document_embeddings = model.encode(
    documents,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

dimension = document_embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(document_embeddings.astype("float32"))

print("FAISS documents:", index.ntotal)

Batches:   0%|          | 0/162 [00:00<?, ?it/s]

FAISS documents: 5183


In [13]:
claim = queries["0"]

candidates = retrieve_candidates(claim, top_k=20)

reranked_results = rerank_documents(
    claim,
    candidates,
    top_k=5
)

for i, result in enumerate(reranked_results, 1):
    print(f"\nRank: {i}")
    print("Document ID:", result["doc_id"])
    print("Title:", result["title"])
    print("Retrieval score:", result["retrieval_score"])
    print("Reranking score:", result["rerank_score"])


Rank: 1
Document ID: 25985964
Title: Very small embryonic-like stem cells with maximum regenerative potential get discarded during cord blood banking and bone marrow processing for autologous stem cell therapy.
Retrieval score: 0.25168028473854065
Reranking score: -9.417900085449219

Rank: 2
Document ID: 4346436
Title: Nonlinear Elasticity in Biological Gels
Retrieval score: 0.333349347114563
Reranking score: -9.974836349487305

Rank: 3
Document ID: 7583104
Title: IDEAL in meshes for prolapse, urinary incontinence, and hernia repair.
Retrieval score: 0.23534882068634033
Reranking score: -10.431892395019531

Rank: 4
Document ID: 2121272
Title: Stresses at the cell-to-substrate interface during locomotion of fibroblasts
Retrieval score: 0.23986881971359253
Reranking score: -10.561408996582031

Rank: 5
Document ID: 17388232
Title: Mechanical regulation of cell function with geometrically modulated elastomeric substrates
Retrieval score: 0.3129125237464905
Reranking score: -10.67094230651

In [14]:
def evidence_verification_pipeline(claim, retrieval_k=20, final_k=5):

    candidates = retrieve_candidates(claim, top_k=retrieval_k)

    ranked_documents = rerank_documents(
        claim,
        candidates,
        top_k=final_k
    )

    for document in ranked_documents:

        evidence = document["text"]

        document["verification"] = verify_claim(
            claim,
            evidence
        )

    return ranked_documents

In [16]:
import torch
import torch.nn.functional as F

def verify_claim(claim, evidence):

    inputs = nli_tokenizer(
        evidence,
        claim,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    with torch.no_grad():
        outputs = nli_model(**inputs)

    probabilities = F.softmax(outputs.logits, dim=1)[0]

    labels = nli_model.config.id2label

    results = {
        labels[i].lower(): probabilities[i].item()
        for i in range(len(labels))
    }

    return results

In [17]:
print(verify_claim)

<function verify_claim at 0x7d1b43f31080>


In [19]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

nli_name = "cross-encoder/nli-deberta-v3-base"

nli_tokenizer = AutoTokenizer.from_pretrained(nli_name)

nli_model = AutoModelForSequenceClassification.from_pretrained(
    nli_name
)

nli_model = nli_model.to(device)
nli_model.eval()

print("NLI tokenizer and model loaded successfully!")
print("Labels:", nli_model.config.id2label)

config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B /  738MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

NLI tokenizer and model loaded successfully!
Labels: {0: 'contradiction', 1: 'entailment', 2: 'neutral'}


In [20]:
claim = "Vitamin D supplementation reduces respiratory infections."

evidence = """
Vitamin D supplementation was associated with a reduced risk
of acute respiratory infections in the studied population.
"""

print(verify_claim(claim, evidence))

{'contradiction': 0.00010337658022763208, 'entailment': 0.9950318336486816, 'neutral': 0.004864695016294718}


In [21]:
claim = queries["0"]

results = evidence_verification_pipeline(claim)

for result in results:

    print("\nTitle:", result["title"])
    print("Document ID:", result["doc_id"])
    print("Verification:", result["verification"])


Title: Very small embryonic-like stem cells with maximum regenerative potential get discarded during cord blood banking and bone marrow processing for autologous stem cell therapy.
Document ID: 25985964
Verification: {'contradiction': 0.029762782156467438, 'entailment': 0.00016595808847341686, 'neutral': 0.9700712561607361}

Title: Nonlinear Elasticity in Biological Gels
Document ID: 4346436
Verification: {'contradiction': 0.9986847043037415, 'entailment': 2.757315814960748e-05, 'neutral': 0.001287639606744051}

Title: IDEAL in meshes for prolapse, urinary incontinence, and hernia repair.
Document ID: 7583104
Verification: {'contradiction': 0.013622092083096504, 'entailment': 0.0003353079955559224, 'neutral': 0.9860426783561707}

Title: Stresses at the cell-to-substrate interface during locomotion of fibroblasts
Document ID: 2121272
Verification: {'contradiction': 0.9992004036903381, 'entailment': 2.155689799110405e-05, 'neutral': 0.0007779946899972856}

Title: Mechanical regulation o